[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oalnaseri/mobilcom_course/blob/main/04_satellite_slant_range_interactive.ipynb)

> **Run this notebook in Google Colab** — click the badge above (no local install needed).
> The setup cell below installs dependencies and enables interactive `ipywidgets` sliders in Colab.


# 🛰️ Satellite Slant Range, Path Loss & Isoflux Pattern
### Interactive Teaching Notebook — DHBW Mobile Communications
---
Converted from the MATLAB script `slant_range.txt`.

This notebook computes, as a function of the **ground elevation angle** (0° = horizon, 90° = zenith):
1. **Slant range** — the Tx–satellite distance (law of cosines geometry)
2. **Free-space path loss** vs. elevation angle
3. **Path loss vs. θ** (elevation seen from the *satellite* antenna)
4. **Isoflux shaping function** — the desired antenna-gain pattern that compensates path loss

> **Equations:** *Satellite Communications Systems* — (2.45c) into (2.45b), result in (2.40).


In [ ]:
# ============================================================
#  COLAB / ENVIRONMENT SETUP  (safe to run locally too)
# ============================================================
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # numpy + matplotlib ship with Colab; ensure ipywidgets is present
    !pip install -q ipywidgets

    # Required so ipywidgets sliders / interactive output render in Colab
    from google.colab import output
    output.enable_custom_widget_manager()
    print('✅ Colab detected — custom widget manager enabled.')
else:
    print('✅ Running locally (Jupyter) — no extra setup needed.')

# Inline backend works reliably for slider-driven redraws in both envs
%matplotlib inline


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')
print("✅ Libraries loaded.")


In [ ]:
# PHYSICS — slant range, path loss, satellite-side angle theta
c0 = 3e8  # speed of light [m/s]

def compute_slant_geometry(frequency_Hz, height_above_surface_m, radius_Earth_m,
                            elev_deg=np.arange(0, 91)):
    """
    Returns dict with elevation grid, slant range [m], path loss [dB],
    theta (satellite-antenna elevation) [deg], and isoflux shaping function [dBi].
    """
    elev_rad = np.deg2rad(elev_deg)
    r = radius_Earth_m
    d_centre = height_above_surface_m + radius_Earth_m   # Tx-to-Earth-centre distance

    # Law of cosines (2.40) with the enclosed angle from (2.45b/c)
    enclosed = (np.pi/2
                - np.arcsin(r * np.cos(elev_rad) / d_centre)
                - elev_rad)
    slant_range_m = np.sqrt(r**2 + d_centre**2 - 2*r*d_centre*np.cos(enclosed))

    # Free-space path loss [dB]
    lam = c0 / frequency_Hz
    path_loss_dB = 10*np.log10((lam / (4*np.pi*slant_range_m))**2)

    # theta = elevation seen from satellite antenna, eq. (2.45c)
    theta_deg = np.rad2deg(np.arcsin(r * np.cos(elev_rad) / d_centre))

    # Isoflux shaping function: gain needed to flatten received flux
    shaping_dBi = np.max(path_loss_dB) - path_loss_dB

    return dict(elev_deg=elev_deg, slant_range_m=slant_range_m,
                path_loss_dB=path_loss_dB, theta_deg=theta_deg,
                shaping_dBi=shaping_dBi, lam=lam, d_centre=d_centre)

print("✅ Slant-range physics defined.")


In [ ]:
# STATIC PLOTS with the MATLAB default parameters
res = compute_slant_geometry(frequency_Hz=1e9,
                             height_above_surface_m=813e3,
                             radius_Earth_m=6370e3)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.patch.set_facecolor('#f5f5f5')

axes[0,0].plot(res['elev_deg'], res['slant_range_m']/1e3, color='#2980b9', lw=2)
axes[0,0].set_xlabel('Elevation angle at ground vs. horizon [deg]')
axes[0,0].set_ylabel('Slant range [km]')
axes[0,0].set_title('Slant Range vs. Elevation', fontweight='bold')

axes[0,1].plot(res['elev_deg'], res['path_loss_dB'], color='#e74c3c', lw=2)
axes[0,1].set_xlabel('Elevation angle at ground vs. horizon [deg]')
axes[0,1].set_ylabel('Path loss [dB]')
axes[0,1].set_title('Path Loss vs. Ground Elevation', fontweight='bold')

axes[1,0].plot(res['theta_deg'], res['path_loss_dB'], color='#8e44ad', lw=2)
axes[1,0].set_xlabel('Theta [deg] (elevation seen from satellite antenna)')
axes[1,0].set_ylabel('Path loss [dB]')
axes[1,0].set_title('Path Loss vs. Satellite-Antenna Angle', fontweight='bold')

axes[1,1].plot(res['theta_deg'], res['shaping_dBi'], color='#27ae60', lw=2)
axes[1,1].set_xlabel('Theta [deg] (elevation seen from satellite antenna)')
axes[1,1].set_ylabel('Shaping function for antenna gain [dBi]')
axes[1,1].set_title('Isoflux Desired Pattern', fontweight='bold')

for ax in axes.flat:
    ax.grid(True, linestyle=':', alpha=0.5)
    ax.set_facecolor('#eaf4fb')

plt.tight_layout()
plt.show()


In [ ]:
# INTERACTIVE EXPLORER
style  = {'description_width': '190px'}
layout = widgets.Layout(width='470px')

sl_freq = widgets.FloatSlider(value=1.0, min=0.5, max=30.0, step=0.5,
    description='Frequency [GHz]:', style=style, layout=layout, readout_format='.1f')
sl_alt  = widgets.FloatSlider(value=813, min=300, max=36000, step=1,
    description='Orbit altitude [km]:', style=style, layout=layout, readout_format='.0f')
sl_re   = widgets.FloatSlider(value=6370, min=6000, max=6400, step=10,
    description='Earth radius [km]:', style=style, layout=layout, readout_format='.0f')

out = widgets.Output()

def update(_=None):
    with out:
        clear_output(wait=True)
        res = compute_slant_geometry(frequency_Hz=sl_freq.value*1e9,
                                     height_above_surface_m=sl_alt.value*1e3,
                                     radius_Earth_m=sl_re.value*1e3)
        fig, axes = plt.subplots(2, 2, figsize=(14, 9))
        fig.patch.set_facecolor('#f5f5f5')

        axes[0,0].plot(res['elev_deg'], res['slant_range_m']/1e3, color='#2980b9', lw=2)
        axes[0,0].set_xlabel('Ground elevation vs. horizon [deg]')
        axes[0,0].set_ylabel('Slant range [km]')
        axes[0,0].set_title('Slant Range vs. Elevation', fontweight='bold')

        axes[0,1].plot(res['elev_deg'], res['path_loss_dB'], color='#e74c3c', lw=2)
        axes[0,1].set_xlabel('Ground elevation vs. horizon [deg]')
        axes[0,1].set_ylabel('Path loss [dB]')
        axes[0,1].set_title('Path Loss vs. Ground Elevation', fontweight='bold')

        axes[1,0].plot(res['theta_deg'], res['path_loss_dB'], color='#8e44ad', lw=2)
        axes[1,0].set_xlabel('Theta [deg] (from satellite antenna)')
        axes[1,0].set_ylabel('Path loss [dB]')
        axes[1,0].set_title('Path Loss vs. Satellite-Antenna Angle', fontweight='bold')

        axes[1,1].plot(res['theta_deg'], res['shaping_dBi'], color='#27ae60', lw=2)
        axes[1,1].set_xlabel('Theta [deg] (from satellite antenna)')
        axes[1,1].set_ylabel('Shaping gain [dBi]')
        axes[1,1].set_title('Isoflux Desired Pattern', fontweight='bold')

        for ax in axes.flat:
            ax.grid(True, linestyle=':', alpha=0.5)
            ax.set_facecolor('#eaf4fb')

        plt.suptitle(f"f = {sl_freq.value:.1f} GHz   |   altitude = {sl_alt.value:.0f} km   |"
                     f"   λ = {res['lam']*100:.2f} cm",
                     fontsize=12, fontweight='bold', y=1.01)
        plt.tight_layout()
        plt.show()

        sr = res['slant_range_m']
        print(f"  Slant range @ horizon (0°)  = {sr[0]/1e3:8.1f} km")
        print(f"  Slant range @ zenith  (90°) = {sr[-1]/1e3:8.1f} km  (= altitude)")
        print(f"  Max theta (satellite side)  = {res['theta_deg'][0]:6.2f}°")
        print(f"  Path-loss spread            = {res['shaping_dBi'].max():6.2f} dB (isoflux gain range)")

for sl in [sl_freq, sl_alt, sl_re]:
    sl.observe(update, names='value')

ui = widgets.VBox([
    widgets.HTML("<h3>⚙️ Satellite Geometry Parameters</h3>"),
    widgets.HBox([widgets.VBox([sl_freq, sl_alt]), widgets.VBox([sl_re])]),
    out
])
display(ui)
update()


## 📚 Theory Reference

### Slant-range geometry (law of cosines)
With Earth radius $R$, orbit altitude $h$, and ground elevation angle $\varepsilon$:

$$d_{centre} = R + h$$

$$\theta = \arcsin\!\left(\frac{R\cos\varepsilon}{R+h}\right) \quad\text{(elevation seen from satellite, eq. 2.45c)}$$

$$s = \sqrt{R^2 + (R+h)^2 - 2R(R+h)\cos\!\left(\tfrac{\pi}{2} - \theta - \varepsilon\right)}$$

### Free-space path loss
$$L(s) = 10\log_{10}\!\left(\frac{\lambda}{4\pi s}\right)^2 \qquad \lambda = \frac{c_0}{f}$$

### Isoflux shaping function
To deliver constant flux on the ground, the satellite antenna gain must **rise toward the horizon** (larger slant range) exactly compensating the extra path loss:

$$G_{shape}(\theta) = \max_\varepsilon L - L(\theta) \quad [\text{dBi}]$$

> **Key insight:** At the horizon ($\varepsilon = 0$) the slant range is largest → highest path loss → the antenna needs the **most** gain there. At zenith the slant range equals the orbit altitude.
